In [0]:
#%run ./env ----- A decommenter pour lancer le notebook separement
#%run ./python_libraries ----- A decommenter pour lancer le notebook separement
#%run ../delta_function ----- A decommenter pour lancer le notebook separement

In [0]:
%run ../env

In [0]:
%run ../python_libraries 

In [0]:
%run ../../delta_function

In [0]:
%run ./load_data 

## Construction dim_calendar
Une ligne = un jour. Cle primaire = day_date.
Ecrite dans le schema common (partagee entre tous les rapports).

TODO: ajuster date_debut si besoin d'un historique plus long.
Va jusqu'au 31/12 de l'annee suivante pour couvrir les batches planifies dans le futur
(et eviter qu'un arret du job coupe les derniers mois dans les rapports).

In [0]:
date_debut = "2023-01-01"

dim_calendar = (
    spark.createDataFrame([(1,)], ["dummy"])
    .select(
        F.explode(
            F.sequence(
                F.to_date(F.lit(date_debut)),
                F.expr("make_date(year(current_date()) + 1, 12, 31)"),
                F.expr("interval 1 day")
            )
        ).alias("day_date")
    )
)

dim_calendar = (
    dim_calendar
    .withColumn("year", F.year("day_date"))
    .withColumn("month_num", F.month("day_date"))
    .withColumn(
        "month_name_fr",
        F.when(F.col("month_num") == 1, "Janvier")
         .when(F.col("month_num") == 2, "Fevrier")
         .when(F.col("month_num") == 3, "Mars")
         .when(F.col("month_num") == 4, "Avril")
         .when(F.col("month_num") == 5, "Mai")
         .when(F.col("month_num") == 6, "Juin")
         .when(F.col("month_num") == 7, "Juillet")
         .when(F.col("month_num") == 8, "Aout")
         .when(F.col("month_num") == 9, "Septembre")
         .when(F.col("month_num") == 10, "Octobre")
         .when(F.col("month_num") == 11, "Novembre")
         .when(F.col("month_num") == 12, "Decembre")
    )
    .withColumn("quarter", F.quarter("day_date"))
    .withColumn("week_num", F.weekofyear("day_date"))
    .withColumn("day_of_week_num", F.dayofweek("day_date"))
    .withColumn(
        "day_name_fr",
        F.when(F.col("day_of_week_num") == 1, "Dimanche")
         .when(F.col("day_of_week_num") == 2, "Lundi")
         .when(F.col("day_of_week_num") == 3, "Mardi")
         .when(F.col("day_of_week_num") == 4, "Mercredi")
         .when(F.col("day_of_week_num") == 5, "Jeudi")
         .when(F.col("day_of_week_num") == 6, "Vendredi")
         .when(F.col("day_of_week_num") == 7, "Samedi")
    )
    .withColumn("year_week", F.concat_ws("-S", F.col("year"), F.lpad(F.col("week_num"), 2, "0")))
    .withColumn("year_month", F.date_format("day_date", "yyyy-MM"))
)

## Ecriture Delta

In [0]:
current_process = "dim_calendar"
target_dim_calendar = current_catalog + "." + common_schema + "." + current_process
print(target_dim_calendar)

In [0]:
all_columns = dim_calendar.columns
primary_key = ['day_date']
additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    display(all_columns)
    print(additional_columns)

In [0]:
handle_table_update(
    dim_calendar,
    target_dim_calendar,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode
)